In [1]:
import requests
import time
import pandas as pd
from datetime import datetime, timedelta

In [2]:
API_KEY = "qrZUHabbbGHJWU95326BITrpe1ZX6SbC79MvFmbIKuEICM9l"

MAX_MINUTE_RETRIES = 3   # retry attempts for minute limit
MINUTE_SLEEP = 60        # NYT allows ~5 requests per minute

def fetch_articles(query, begin_date, end_date, page):

    url = "https://api.nytimes.com/svc/search/v2/articlesearch.json"

    params = {
        "api-key": API_KEY,
        "q": query,
        "begin_date": begin_date,
        "end_date": end_date,
        "page": page,
    }

    retries = 0

    while retries <= MAX_MINUTE_RETRIES:
        response = requests.get(url, params=params)

        if response.status_code == 200:
            data = response.json()
            return data.get("response", {}).get("docs", [])

        if response.status_code == 429:
            retries += 1

            if retries > MAX_MINUTE_RETRIES:
                print("Daily rate limit likely reached.")
                return None  # signal to stop everything

            print(f"Minute rate limit hit. Sleeping {MINUTE_SLEEP}s...")
            time.sleep(MINUTE_SLEEP)
            continue
    
        print("Error:", response.status_code)
        return []

    return None


def get_all_articles(query, start_date, end_date):

    all_articles = []
    current_date = start_date
    stop_collection = False

    while current_date <= end_date and not stop_collection:

        month_end = (current_date.replace(day=28) + timedelta(days=4)).replace(day=1) - timedelta(days=1)
        if month_end > end_date:
            month_end = end_date

        begin_str = current_date.strftime("%Y%m%d")
        end_str = month_end.strftime("%Y%m%d")

        print(f"Fetching {begin_str} to {end_str}")

        month_articles = []          # store articles for THIS month only
        month_failed = False         # track if any request failed

        for page in range(5):

            docs = fetch_articles(query, begin_str, end_str, page)

            if docs is None:
                month_failed = True  # mark month as incomplete
                stop_collection = True
                break

            if not docs:
                break

            month_articles.extend(docs)
            
            time.sleep(12)

        # Only commit the month if ALL requests succeeded
        if not month_failed:
            all_articles.extend(month_articles)
        else:
            print(f"Skipping incomplete month {begin_str}-{end_str}")

        current_date = month_end + timedelta(days=1)

    print(f"Returning {len(all_articles)} articles collected so far.")
    return pd.json_normalize(all_articles)

In [3]:
# Run it
start = datetime(2014, 5, 1)
end = datetime(2026, 2, 28)

df = get_all_articles("economy", start, end)

df.head()

Fetching 20140501 to 20140531
Fetching 20140601 to 20140630
Fetching 20140701 to 20140731
Fetching 20140801 to 20140831
Fetching 20140901 to 20140930
Fetching 20141001 to 20141031
Fetching 20141101 to 20141130
Fetching 20141201 to 20141231
Fetching 20150101 to 20150131
Fetching 20150201 to 20150228
Fetching 20150301 to 20150331
Fetching 20150401 to 20150430
Fetching 20150501 to 20150531
Fetching 20150601 to 20150630
Fetching 20150701 to 20150731
Fetching 20150801 to 20150831
Fetching 20150901 to 20150930
Fetching 20151001 to 20151031
Fetching 20151101 to 20151130
Fetching 20151201 to 20151231
Fetching 20160101 to 20160131
Fetching 20160201 to 20160229
Fetching 20160301 to 20160331
Fetching 20160401 to 20160430
Fetching 20160501 to 20160531
Fetching 20160601 to 20160630
Fetching 20160701 to 20160731
Fetching 20160801 to 20160831
Fetching 20160901 to 20160930
Fetching 20161001 to 20161031
Fetching 20161101 to 20161130
Fetching 20161201 to 20161231
Fetching 20170101 to 20170131
Fetching 2

,abstract,document_type,_id,keywords,news_desk,print_page,print_section,pub_date,section_name,snippet,...,headline.kicker,headline.print_headline,multimedia.caption,multimedia.credit,multimedia.default.url,multimedia.default.height,multimedia.default.width,multimedia.thumbnail.url,multimedia.thumbnail.height,multimedia.thumbnail.width
0,Uncle Sam faces a new heavyweight.,article,nyt://article/40b433f9-694e-54df-8f6d-70547b8a...,"[{'name': 'Location', 'value': 'China', 'rank'...",OpEd,,,2014-05-03T18:03:16Z,Opinion,Uncle Sam faces a new heavyweight.,...,Heng,,,Heng,https://static01.nyt.com/images/2014/05/02/opi...,433,600,https://static01.nyt.com/images/2014/05/02/opi...,75,75
1,The economy contracted at an annual rate of 1 ...,article,nyt://article/e0be55ae-4826-5052-a038-48f91c07...,"[{'name': 'Subject', 'value': 'United States E...",Business,1,B,2014-05-29T12:33:02Z,Business,The economy contracted at an annual rate of 1 ...,...,,U.S. Economy Contracted in a Frigid First Quarter,"Workers installing a roof in Broomfield, Colo....",Rick Wilking/Reuters,https://static01.nyt.com/images/2014/05/30/bus...,400,600,https://static01.nyt.com/images/2014/05/30/bus...,75,75
2,The better-than-expected numbers for employmen...,article,nyt://article/b689f7b0-8b14-55dd-bc14-fd6613be...,"[{'name': 'Subject', 'value': 'United States E...",Business,1,B,2014-05-02T00:35:38Z,Business,The better-than-expected numbers for employmen...,...,,Spending and Pay Gains Suggest Better Economy,Shoppers like these in New York helped to incr...,Andrew Burton/Getty Images,https://static01.nyt.com/images/2014/05/02/bus...,322,600,https://static01.nyt.com/images/2014/05/02/bus...,75,75
3,"After a slowdown at the start of the year, the...",article,nyt://article/bea4df85-c5ab-5822-81a1-fff13434...,"[{'name': 'Subject', 'value': 'Labor and Jobs'...",Business,1,A,2014-05-02T12:31:48Z,Business,"After a slowdown at the start of the year, the...",...,,Jump in Payrolls Is Seen as a Sign of New Opti...,A job fair in Detroit last month.,Joshua Lott/Getty Images,https://static01.nyt.com/images/2014/05/03/bus...,400,600,https://static01.nyt.com/images/2014/05/03/bus...,75,75
4,The excess profits companies can extract from ...,article,nyt://article/2955bba0-59e5-52f1-bba8-a03d9049...,"[{'name': 'Person', 'value': 'Porter, Eduardo'...",Business,1,B,2014-05-28T00:47:18Z,Business,The excess profits companies can extract from ...,...,,Concentrated Markets Take Big Toll on Economy,Eduardo Porter,Earl Wilson/The New York Times,https://static01.nyt.com/images/2012/02/29/bus...,900,600,https://static01.nyt.com/images/2012/02/29/bus...,75,75


In [4]:
df.to_csv("nytimes_economy_2.csv", index=False)